# Main Cloud Orchestrator - Temporal Pseudo-Labeling

This notebook serves as the main orchestrator for running the Fase 0 (Pseudo-Labeling) pipeline on Google Colab or Kaggle. It performs a sparse checkout of the lightweight `experiments/` directory, installs the dependencies, runs the unit tests, and triggers the class-based pseudo-labeling pipeline. All results are synced directly to Google Drive. Once finished (successfully or on error), it automatically disconnects the VM runtime to save credits.

## Cell 1: Shallow Clone and Sparse Checkout
Clones only the lightweight `experiments/` code directory (ignoring heavy dataset or model directories) and installs the package in editable mode.

In [ ]:
import os
from pathlib import Path

REPO_NAME = 'ia_article'
REPO_URL = 'https://github.com/unsa-semester-2026-A/ia_article.git'

# 1. Clone repository sparsely
if not os.path.exists(REPO_NAME):
    print(f"Clonando {REPO_NAME} (solo directorio 'experiments' y sin historial)...")
    !git clone -q --depth 1 --filter=blob:none --sparse {REPO_URL}
    %cd {REPO_NAME}
    !git sparse-checkout set experiments
    %cd experiments
else:
    print(f"Actualizando repositorio {REPO_NAME}...")
    %cd {REPO_NAME}
    !git pull -q
    %cd experiments

# 2. Verify working directory
current_dir = Path(os.getcwd())
if current_dir.name != 'experiments':
    raise RuntimeError(f"Fallo al navegar al directorio. Ruta actual: {current_dir}")

# 3. Install dependencies in editable mode
print("Instalando el paquete en modo editable con dependencias [cloud]...")
%pip install -q -e .[cloud]

## Cell 2: Mount Google Drive
Mounts your Google Drive account to access raw CSV/ZIP datasets and API token configs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 3: Run pytest Unit Tests
Executes the colocated unit tests in the repository to guarantee all functions and packages compile properly before starting.

In [ ]:
!pytest src/

## Cell 4: Run Pseudo-Labeler Pipeline & Release Resources
Triggers the class-based production pseudo-labeling pipeline. All output clip JSONs are written locally in local VM RAM and uploaded to the Drive checkpoints subfolder (`for_each_clip`) dynamically after each iteration. Resuming is handled natively via Drive queries. Upon completion (or failure), the Google Colab VM is automatically disconnected to save compute credits.

In [ ]:
# Runs the class-based pipeline using configured defaults
%run src/pseudo_labeling/pseudo_labeler.py